# Problem Inputs

In [ ]:
### DATA
TIME_HORIZON = 24  # hours
TIME_UNIT = 1      # 1 hour per slot
ENERGY_PRICE = { # Electricity price (¥ / kWh) by hour
    0: 8, 1: 8, 2: 8, 3: 8, 4: 8, 5: 8,          # off-peak
    6: 18, 7: 18, 8: 18, 9: 18,                  # morning peak
    10: 12, 11: 12, 12: 12, 13: 12, 14: 12, 15: 12, 16: 12,
    17: 22, 18: 22, 19: 22, 20: 22, 21: 22,      # evening peak
    22: 10, 23: 10                               # late night
}

### MACHINES (capacity = 1)
SERVERS = {
    "S1": {"type": "CPU",     "power_kw": 2.0},
    "S2": {"type": "CPU",     "power_kw": 2.0},
    "S3": {"type": "GPU",     "power_kw": 4.5},
    "S4": {"type": "Storage","power_kw": 1.2},
}

### JOBS (Government Workload)
JOBS = {
    "J1": {
        "description": "Emergency response analytics",
        "priority": "HIGH",
        "deadline": 8,
        "delay_weight": 10,
        "operations": [
            {"server_type": "Storage", "duration": 1},
            {"server_type": "CPU",     "duration": 2},
            {"server_type": "GPU",     "duration": 1},
        ],
    },

    "J2": {
        "description": "Disaster damage estimation",
        "priority": "HIGH",
        "deadline": 10,
        "delay_weight": 8,
        "operations": [
            {"server_type": "Storage", "duration": 1},
            {"server_type": "GPU",     "duration": 2},
            {"server_type": "CPU",     "duration": 1},
        ],
    },

    "J3": {
        "description": "Tax data aggregation",
        "priority": "MEDIUM",
        "deadline": 18,
        "delay_weight": 4,
        "operations": [
            {"server_type": "CPU",     "duration": 2},
            {"server_type": "Storage", "duration": 1},
            {"server_type": "CPU",     "duration": 1},
        ],
    },

    "J4": {
        "description": "Census preprocessing",
        "priority": "MEDIUM",
        "deadline": 20,
        "delay_weight": 3,
        "operations": [
            {"server_type": "Storage", "duration": 1},
            {"server_type": "CPU",     "duration": 2},
        ],
    },

    "J5": {
        "description": "Open data batch update",
        "priority": "LOW",
        "deadline": 22,
        "delay_weight": 1,
        "operations": [
            {"server_type": "CPU",     "duration": 1},
            {"server_type": "Storage", "duration": 1},
        ],
    },

    "J6": {
        "description": "Archive and backup",
        "priority": "LOW",
        "deadline": 24,
        "delay_weight": 0.5,
        "operations": [
            {"server_type": "Storage", "duration": 2},
            {"server_type": "CPU",     "duration": 2},
        ],
    },
}






SyntaxError: unterminated string literal (detected at line 52) (4263212160.py, line 52)

# Decision Variable

In [ ]:
SCHEDULE = {
    "J1": [
        {"server": "S4", "start": 0},  # Storage
        {"server": "S1", "start": 1},  # CPU
        {"server": "S3", "start": 3},  # GPU
    ],
    "J2": [
        {"server": "S4", "start": 1},
        {"server": "S3", "start": 2},
        {"server": "S2", "start": 4},
    ],
    # ...
}


# Evaluation

- total delay
- energy purchased cost

In [ ]:
OBJECTIVE_WEIGHTS = {
    "energy_cost": 1.0,
    "delay_penalty": 5.0,
}

def energy_cost(power_kw, start_time, duration):
    """Compute energy cost for an operation."""
    cost = 0.0
    for t in range(start_time, start_time + duration):
        cost += power_kw * ENERGY_PRICE[t % TIME_HORIZON]
    return cost

def delay_penalty(completion_time, deadline, weight):
    return max(0, completion_time - deadline) * weight

def evaluate_schedule(schedule, jobs, servers, energy_price, weights):
    """
    Evaluates a job-shop schedule with energy and delay costs.
    Returns a detailed report dictionary.
    """

    server_usage = {s: [] for s in servers}
    job_completion_times = {}
    total_energy_cost = 0.0
    total_delay_penalty = 0.0
    violations = []

    # --- Check each job ---
    for job_id, ops in schedule.items():
        job_data = jobs[job_id]
        job_ops = job_data["operations"]

        if len(ops) != len(job_ops):
            violations.append(f"{job_id}: wrong number of operations")
            continue

        prev_end = None

        for idx, op in enumerate(ops):
            server_id = op["server"]
            start = op["start"]
            duration = job_ops[idx]["duration"]
            end = start + duration

            # --- Job order constraint ---
            if prev_end is not None and start < prev_end:
                violations.append(
                    f"{job_id}: operation {idx} starts before previous ends"
                )

            prev_end = end

            # --- Server type constraint ---
            required_type = job_ops[idx]["server_type"]
            actual_type = servers[server_id]["type"]
            if required_type != actual_type:
                violations.append(
                    f"{job_id}: operation {idx} requires {required_type}, "
                    f"got {actual_type}"
                )

            # --- Record server usage ---
            server_usage[server_id].append(
                (start, end, job_id, idx)
            )

            # --- Energy cost ---
            power = servers[server_id]["power_kw"]
            for t in range(start, end):
                total_energy_cost += power * energy_price[t]

        # --- Completion time & delay ---
        completion = prev_end
        job_completion_times[job_id] = completion

        deadline = job_data["deadline"]
        weight = job_data["delay_weight"]
        delay = max(0, completion - deadline)
        total_delay_penalty += delay * weight

    # --- Check server capacity constraints ---
    for server_id, intervals in server_usage.items():
        intervals.sort()
        for i in range(len(intervals) - 1):
            _, end1, job1, _ = intervals[i]
            start2, _, job2, _ = intervals[i + 1]
            if start2 < end1:
                violations.append(
                    f"{server_id}: overlap between {job1} and {job2}"
                )

    # --- Final objective ---
    objective_value = (
        weights["energy_cost"] * total_energy_cost
        + weights["delay_penalty"] * total_delay_penalty
    )

    return {
        "objective": objective_value,
        "energy_cost": total_energy_cost,
        "delay_penalty": total_delay_penalty,
        "job_completion_times": job_completion_times,
        "violations": violations,
    }

report = evaluate_schedule(
    schedule=SCHEDULE,
    jobs=JOBS,
    servers=SERVERS,
    energy_price=ENERGY_PRICE,
    weights=OBJECTIVE_WEIGHTS,
)

print("Objective:", report["objective"])
print("Energy cost:", report["energy_cost"])
print("Delay penalty:", report["delay_penalty"])
print("Violations:", report["violations"])

# Initial Strategies
1) FIFO
2) Energy Greedy
3) Optimized Heuristic

In [ ]:
def fifo_scheduler(jobs, servers, time_horizon=24):
    """
    First-In-First-Out (FIFO) job-shop scheduler.

    Returns:
        schedule: dict compatible with evaluate_schedule()
    """

    # Initialize availability
    server_available_time = {s: 0 for s in servers}
    job_available_time = {j: 0 for j in jobs}

    schedule = {}

    # FIFO order: sorted by job ID
    for job_id in sorted(jobs.keys()):
        job_data = jobs[job_id]
        schedule[job_id] = []

        for op in job_data["operations"]:
            required_type = op["server_type"]
            duration = op["duration"]

            # Find all compatible servers
            compatible_servers = [
                s for s, data in servers.items()
                if data["type"] == required_type
            ]

            # Choose the server that becomes available earliest
            chosen_server = min(
                compatible_servers,
                key=lambda s: server_available_time[s]
            )

            # Earliest possible start
            start_time = max(
                job_available_time[job_id],
                server_available_time[chosen_server]
            )

            # Assign operation
            schedule[job_id].append({
                "server": chosen_server,
                "start": start_time
            })

            # Update availability
            finish_time = start_time + duration
            job_available_time[job_id] = finish_time
            server_available_time[chosen_server] = finish_time

    return schedule

fifo_schedule = fifo_scheduler(JOBS, SERVERS)
report = evaluate_schedule(
    schedule=fifo_schedule,
    jobs=JOBS,
    servers=SERVERS,
    energy_price=ENERGY_PRICE,
    weights=OBJECTIVE_WEIGHTS,
)

print("FIFO Objective:", report["objective"])
print("FIFO Energy Cost:", report["energy_cost"])
print("FIFO Delay Penalty:", report["delay_penalty"])
print("Violations:", report["violations"])

In [ ]:
def energy_greedy_scheduler(jobs, servers, energy_price, time_horizon=24):
    """
    Energy-greedy job-shop scheduler.
    Schedules operations in low-energy-cost time slots, ignoring deadlines.
    """

    server_available_time = {s: 0 for s in servers}
    job_available_time = {j: 0 for j in jobs}

    schedule = {}

    def operation_energy_cost(server_id, start, duration):
        power = servers[server_id]["power_kw"]
        cost = 0.0
        for t in range(start, start + duration):
            cost += power * energy_price[t % time_horizon]
        return cost

    # Fixed job order (FIFO job order, energy-aware timing)
    for job_id in sorted(jobs.keys()):
        job_data = jobs[job_id]
        schedule[job_id] = []

        for op in job_data["operations"]:
            required_type = op["server_type"]
            duration = op["duration"]

            compatible_servers = [
                s for s, data in servers.items()
                if data["type"] == required_type
            ]

            best_choice = None
            best_cost = float("inf")

            for server_id in compatible_servers:
                earliest_start = max(
                    job_available_time[job_id],
                    server_available_time[server_id]
                )

                # Try shifting the start time within the horizon
                for start in range(earliest_start, time_horizon - duration + 1):
                    cost = operation_energy_cost(server_id, start, duration)

                    if cost < best_cost:
                        best_cost = cost
                        best_choice = (server_id, start)

            chosen_server, start_time = best_choice

            schedule[job_id].append({
                "server": chosen_server,
                "start": start_time
            })

            finish_time = start_time + duration
            job_available_time[job_id] = finish_time
            server_available_time[chosen_server] = finish_time

    return schedule

energy_schedule = energy_greedy_scheduler(
    jobs=JOBS,
    servers=SERVERS,
    energy_price=ENERGY_PRICE
)

report = evaluate_schedule(
    schedule=energy_schedule,
    jobs=JOBS,
    servers=SERVERS,
    energy_price=ENERGY_PRICE,
    weights=OBJECTIVE_WEIGHTS,
)

print("Energy-Greedy Objective:", report["objective"])
print("Energy Cost:", report["energy_cost"])
print("Delay Penalty:", report["delay_penalty"])
print("Violations:", report["violations"])